In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir(os.path.dirname(os.getcwd()))

In [3]:
import pandas as pd
import numpy as np

from src.model.similarity import Similarity_Score

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [4]:
path = "data"

In [5]:
sim = Similarity_Score(path=path)

/Users/ritisha/projects/simple_recommendation/src/model/similarity.py:13: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  self.books_df = pd.read_csv(self.path + "/preprocessed/books_cleansed.csv")


In [32]:
feat_df = pd.read_csv(f"{path}/features/feat_data.csv")
recommendations = sim.predict(feat_df)

/var/folders/nb/j0fz9mcs1lv5q2y83pmm1rfc0000gp/T/ipykernel_1164/3064751479.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  feat_df = pd.read_csv(f"{path}/features/feat_data.csv")


5


In [30]:
import pickle
def user_item_similarity(df, user_rating_count_thr=7, book_rating_count_thr=10):

    filtered_df = df[
        (df["user_rating_count"] >= user_rating_count_thr)
        & (df["book_rating_count"] >= book_rating_count_thr)
    ]
    pivot_df = filtered_df.pivot(
        index="isbn", columns="user_id", values="book_rating"
    ).fillna(0)

    with open(f"{path}/features/index_data_meta.pkl", "wb") as f:
        pickle.dump((pivot_df.index), f)

    sc = StandardScaler()
    filtered_mat = sc.fit_transform(pivot_df)

    sim_mat = cosine_similarity(filtered_mat)
    sim_mat[np.arange(sim_mat.shape[0])[:, None] >= np.arange(sim_mat.shape[1])] = (
        np.nan
    )
    np.save(f"{path}/features/sim_mat.npy", sim_mat)

In [21]:
feat_df = pd.read_csv(f"{path}/features/feat_data.csv")
user_item_similarity(feat_df)

/var/folders/nb/j0fz9mcs1lv5q2y83pmm1rfc0000gp/T/ipykernel_1164/1662996389.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  feat_df = pd.read_csv(f"{path}/features/feat_data.csv")


NameError: name 'user_item_similarity' is not defined

In [20]:
import pickle
sim_mat = np.load(f"{path}/features/sim_mat.npy")
with open(f"{path}/features/index_data_meta.pkl", "rb") as f:
    idx = pickle.load(f)

In [18]:
top_idx = sim_mat.argsort()
# [-5: :][::-1][0]
# bookisbn_top = [idx_list[i] for i in top_idx]

array([[ 2818,  5935,  2270, ..., 16531, 14647,     0],
       [ 2818,  2270, 13948, ...,  1999,     1,     0],
       [ 2818,  2270, 11906, ...,     1,     2,     0],
       ...,
       [18316, 18317, 18314, ...,  6101,  6107,  9158],
       [18317, 12202, 12203, ...,  6102,  6108,  9158],
       [    0, 12202, 12203, ...,  6102,  6108, 18317]])

In [63]:
list(idx).index('B00009EF82')

18317

In [134]:
exclude = [list(idx).index(id) for id in ['000000000', '9726106141']]
idx_rec = [i for i in range(sim_mat.shape[0]) if i not in exclude] 

In [66]:
sim_mat.shape[0]

18318

In [135]:
idx_list = list(idx)

In [149]:
all_isbns_idx = {}
for i in ['000000000', '9726106141']:
    i_idx = idx_list.index(i)
    idx_tmp = sim_mat[idx_rec, i_idx].argsort()[-10:][::-1].tolist()
    all_isbns_idx = {j:sim_mat[i_idx,j] for j in idx_tmp}

all_isbns_idx = dict(sorted(all_isbns_idx.items(), key=lambda item: item[1], reverse=True))


bookisbn = [list(idx)[i] for i in  all_isbns_idx.keys()] 

In [150]:
all_isbns_idx.keys()

dict_keys([18313, 18312, 18311, 18310, 18309, 18308, 7103, 4664, 18314, 18315])

In [153]:
sim_mat.argsort()[-10:][::-1][0]

array([    0, 12202, 12203, ...,  6102,  6108, 18317])